# Сагитталь — бинарная классификация (5-fold CV)

Логика обучения и метрик в репозитории: `MLService/training/sagittal_binary_cv.py`, датасеты и K-fold — в `training/datasets/tmj_position_dataset.py`, `training/tmj_position_label_table.py`, метрики — `training/utils/binary_metrics.py`, аугментации 3D — `training/utils/volume_aug_3d.py`.

**Yandex DataSphere:** кропы — подключённый датасет `datasets/tmj/detector_crops_v2` **или** (часто) только **`/home/jupyter/filestore/detector_crops_v2`**, если датасет в проекте не смонтирован (как в `train_binary_position_classifier`). Манифест/метки — в датасете или в `filestore`. Переменные: `TMJ_DATASET_DIR`, **`TMJ_CROP_DIR`** (если кропы нестандартно), `TMJ_MANIFEST_PATH`, `TMJ_LABELS_PATH`, `ML_SERVICE_ROOT`. Инициализация датасета: [init_datasphere_dataset.ipynb](init_datasphere_dataset.ipynb).

План экспериментов: `MLService/docs/superpowers/prompts/improve-sag-classifier-metrics.md`.

**Сплиты:** только train / validation внутри каждого фолда CV, отдельной тестовой выборки нет.

**Дефолты конфига:** backbone `[8,16,32,64]`, `fc_hidden=128`, train-аугментации `strong` (флипы + малые повороты + джиттер яркости).

Здесь: зависимости, `PYTHONPATH`, авто-пути (DataSphere / локально) и запуск `run_sagittal_binary_cv`.

**Отчёт:** в `output_json` пишется полный JSON: по каждому фолду — `epoch_history` (по эпохам: `train_loss`, `val_auc`, метрики на val при пороге 0.5, `lr`), итоговые метрики после порога Youden по train, `n_train_samples` / `n_val_samples`.

**Разбор без остановки обучения:** в конце тетрадки отдельная ячейка читает готовый JSON и пишет txt/json/csv/png — можно запускать в другой тетрадке на том же `filestore`, пока CV ещё идёт (если JSON уже появился; иначе дождись записи).

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "scikit-learn", "nibabel", "tqdm", "scipy"])

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_boot = None
for p in [_HERE, *list(_HERE.parents)[:10], Path("/home/jupyter/project/MasterProject/MLService"), Path("/content/MasterProject/MLService")]:
    if (p / "training" / "sagittal_binary_cv.py").is_file():
        _boot = p.resolve()
        break
if _boot is None:
    _boot = Path("/home/jupyter/project/MasterProject/MLService")
sys.path.insert(0, str(_boot))

from training.utils.datasphere_env import (
    default_cv_output_json,
    infer_mlservice_root,
    is_datasphere,
    sagittal_binary_cv_path_kwargs,
)

MLSERVICE_ROOT = infer_mlservice_root(_HERE)

print("MLService:", MLSERVICE_ROOT)
print("DataSphere:", is_datasphere(), "| crops:", sagittal_binary_cv_path_kwargs()["crop_dir"])

In [ ]:
import os

from training.sagittal_binary_cv import SagittalBinaryCVConfig, run_sagittal_binary_cv, _print_cv_table

path_kw = sagittal_binary_cv_path_kwargs()
out_json = default_cv_output_json(MLSERVICE_ROOT)

cfg = SagittalBinaryCVConfig(
    **path_kw,
    epochs=80,
    batch_size=16,
    num_workers=min(8, max(2, (os.cpu_count() or 8) - 1)),
    train_augment_mode="strong",
    features=(8, 16, 32, 64),
    fc_hidden=128,
    output_json=str(out_json),
    tqdm_disable=False,
)
print("Paths:", path_kw)
print("Output:", out_json)

result = run_sagittal_binary_cv(cfg)
_print_cv_table(result)

### Разбор результатов (отдельная ячейка)

Можно выполнить **после** завершения CV или в **другой** тетрадке на том же `filestore`, не останавливая долгий запуск обучения. Читает тот же JSON (`default_cv_output_json`), печатает сводку и сохраняет рядом: `*_analyze.txt`, `*_analyze_export.json`, при наличии `pandas` — CSV по фолдам и эпохам, при `matplotlib` — `*_analyze_curves.png`.

Если файла JSON ещё нет (старый раннер пишет только после всех фолдов), ячейка сообщит об этом.

При необходимости: `pip install pandas matplotlib`.

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_boot = None
for p in [_HERE, *list(_HERE.parents)[:10], Path("/home/jupyter/project/MasterProject/MLService"), Path("/content/MasterProject/MLService")]:
    if (p / "training" / "sagittal_binary_cv.py").is_file():
        _boot = p.resolve()
        break
if _boot is None:
    _boot = Path("/home/jupyter/project/MasterProject/MLService")
if str(_boot) not in sys.path:
    sys.path.insert(0, str(_boot))

from training.utils.datasphere_env import default_cv_output_json, infer_mlservice_root
from training.sagittal_binary_cv import analyze_sagittal_cv_result

try:
    _mr = MLSERVICE_ROOT
except NameError:
    _mr = infer_mlservice_root(_HERE)

cv_json = default_cv_output_json(_mr)
report_base = cv_json.parent / f"{cv_json.stem}_analyze"

print("CV JSON:", cv_json)
print("Отчёт (префикс имён файлов):", report_base)

if not cv_json.is_file():
    print("Файла ещё нет — дождись конца CV или проверь путь.")
else:
    analyze_sagittal_cv_result(
        json_path=cv_json,
        report_path=report_base,
        show_plots=True,
    )
